# Семинар. Задачи для самостоятельного решения

**Курс:** «Интеллектуальный анализ текстов: от статистических моделей языка до языковых агентов». Семинар к лекции «Нейронные сети: от нейрона до seq2seq с вниманием» (модуль 2).

Десять задач по трём демонстрационным блокнотам: `1_seminar_nejron_i_set`, `2_seminar_tekst_rnn_lstm`, `3_seminar_seq2seq_vnimanie`. Задачи 1–4, 8, 9 расчётные. В задачах 5, 6, 7, 10 нужно изменить готовую модель и объяснить результат.

В ячейках с кодом замените многоточия `...` своим решением. После каждой задачи приведена самопроверка с ожидаемым числом или диапазоном. Вывод по задаче записывается одной-двумя фразами.

Готовые функции и классы импортируются из файла `seminar_nn_utils.py`, который лежит рядом с блокнотом. Это тот же код, что показан в демонстрационных блокнотах. Весь блокнот с решениями выполняется на процессоре за две-три минуты.

In [ ]:
import json
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
torch.set_num_threads(4)

from seminar_nn_utils import *

set_seed()
print("Готово, PyTorch", torch.__version__)

## Задача 1. Нейрон вручную

У нейрона веса $w=(0.5,\,-1.0,\,2.0)$ и смещение $b=0.1$. На вход подан вектор $x=(1.0,\,0.5,\,0.25)$.

1. Вычислите взвешенную сумму $z=w\cdot x+b$.
2. Вычислите выход нейрона для трёх функций активации: сигмоиды, tanh и ReLU.
3. Как изменится выход с ReLU, если смещение станет равным $-1$?

**Самопроверка.** $z=0.6$. Выходы: сигмоида 0.646, tanh 0.537, ReLU 0.6. При $b=-1$ выход ReLU равен 0.

In [ ]:
w = np.array([0.5, -1.0, 2.0])
b = 0.1
x = np.array([1.0, 0.5, 0.25])

z = ...
print("z =", z)
print("сигмоида:", ...)
print("tanh:    ", ...)
print("ReLU:    ", ...)
print("ReLU при b = -1:", ...)

**Вывод:**

_напишите одну-две фразы_

## Задача 2. Softmax и кросс-энтропия

Сеть выдала для трёх классов оценки $z=(2.0,\,1.0,\,0.1)$.

1. Напишите функцию `softmax` и вычислите вероятности классов.
2. Правильным является второй класс (с оценкой 1.0). Вычислите кросс-энтропию $L=-\ln\hat y_c$.
3. Чему были бы равны потери, если бы правильным был первый класс?

**Самопроверка.** Вероятности $(0.659,\,0.242,\,0.099)$, их сумма равна 1. Потери для второго класса 1.417, для первого 0.417.

In [ ]:
def softmax(z):
    ...


z = np.array([2.0, 1.0, 0.1])
p = softmax(z)
print("вероятности:", p, "сумма:", p.sum())
print("потери, правильный класс второй:", ...)
print("потери, правильный класс первый:", ...)

**Вывод:**

_напишите одну-две фразы_

## Задача 3. Логические функции

Нейрон с пороговой активацией выдаёт 1 при $z>0$ и 0 в остальных случаях.

1. Подберите веса $w_1$, $w_2$ и смещение $b$ одного нейрона для функции «$x_1$ И НЕ $x_2$»: ответ равен 1 только на входе $(1,0)$.
2. Сеть для XOR из демонстрационного блокнота: $h_1=\mathrm{ReLU}(x_1+x_2)$, $h_2=\mathrm{ReLU}(x_1+x_2-1)$, $y=h_1-2h_2$. Скрытый слой оставьте прежним и подберите веса $u_1$, $u_2$ и смещение $c$ выходного нейрона $y=u_1h_1+u_2h_2+c$ так, чтобы сеть вычисляла функцию XNOR: ответ 1 на входах $(0,0)$ и $(1,1)$, ответ 0 на входах $(0,1)$ и $(1,0)$.

**Самопроверка.** Ответы первого нейрона на входах $(0,0),(0,1),(1,0),(1,1)$ равны $0,0,1,0$. Ответы сети XNOR равны $1,0,0,1$.

In [ ]:
X4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

# 1. Один нейрон: x1 И НЕ x2
w_and_not, b_and_not = np.array([..., ...]), ...
print("x1 И НЕ x2:", ((X4 @ w_and_not + b_and_not) > 0).astype(int))

# 2. XNOR: скрытый слой прежний, выходной нейрон свой
H = np.maximum(X4 @ np.array([[1, 1], [1, 1]]).T + np.array([0, -1]), 0)
u, c = np.array([..., ...]), ...
print("XNOR:", H @ u + c)

**Вывод:**

_напишите одну-две фразы_

## Задача 4. Граф вычислений

Дана функция $L=(a\cdot b+c)^2$ при $a=2$, $b=-3$, $c=4$.

1. Выполните прямой проход: вычислите промежуточные величины $d=a\cdot b$, $e=d+c$ и значение $L=e^2$.
2. Выполните обратный проход вручную по цепному правилу: найдите $\partial L/\partial a$, $\partial L/\partial b$, $\partial L/\partial c$.
3. Проверьте ответ с помощью `backward` в PyTorch.

**Самопроверка.** $d=-6$, $e=-2$, $L=4$. Производные: $\partial L/\partial a=12$, $\partial L/\partial b=-8$, $\partial L/\partial c=-4$.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b_ = torch.tensor(-3.0, requires_grad=True)
c = torch.tensor(4.0, requires_grad=True)

L = ...
...
print("L =", ...)
print("dL/da, dL/db, dL/dc:", ...)

# расчёт вручную по цепному правилу
print("вручную:", ...)

**Вывод:**

_напишите одну-две фразы_

## Задача 5. Роль активации и размера сети

Данные те же, что в первом блокноте: две изогнутые группы точек. Функции `make_mlp`, `train_mlp`, `mlp_accuracy`, `plot_mlp_boundary` импортированы.

Обучите четыре сети и сравните долю верных ответов на тестовой части:

- 16 нейронов без функции активации (`activation=None`),
- 2 нейрона с ReLU,
- 16 нейронов с ReLU,
- два скрытых слоя по 16 нейронов с ReLU (`n_layers=2`).

Параметры обучения для всех одинаковы: `lr=0.03, epochs=800, optimizer="adam"`. Перед созданием каждой сети вызывайте `set_seed()`. Постройте границы четырёх сетей рядом.

**Самопроверка.** Сеть без активации даёт на тесте от 0.82 до 0.87. Сети с 16 нейронами и ReLU дают не меньше 0.94.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=500, noise=0.22, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)

configs = {"16, без активации": dict(n_hidden=16, activation=None),
           "2, ReLU": ...,
           "16, ReLU": ...,
           "2 слоя по 16, ReLU": ...}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (name, kw) in zip(axes, configs.items()):
    set_seed()
    model = ...
    ...
    plot_mlp_boundary(ax, model, X_train, y_train, f"{name}\nтест: ...")
plt.show()

**Вывод:**

_напишите одну-две фразы_

## Задача 6. Размер эмбеддингов

Классификатор новостей по усреднённым эмбеддингам из второго блокнота. Обучите его при трёх размерах эмбеддинга: 2, 16 и 64 (`emb_dim`), 12 эпох. Сравните долю верных ответов на тесте и число весов в модели. Точка отсчёта, логистическая регрессия на TF-IDF, даёт на этих данных 0.966.

**Самопроверка.** При размере 16 и 64 доля верных ответов на тесте лежит между 0.93 и 0.97. При размере 2 она ниже или равна значению для размера 64. Число весов растёт примерно пропорционально размеру эмбеддинга.

In [ ]:
from sklearn.model_selection import train_test_split

texts, labels = load_news()
T_train, T_test, yn_train, yn_test = train_test_split(texts, labels, test_size=0.25, stratify=labels, random_state=SEED)
tok_train, tok_test = [tokenize(t) for t in T_train], [tokenize(t) for t in T_test]
vocab = build_vocab(tok_train, min_count=3)
ids_train = [[vocab.get(t, 0) for t in d] for d in tok_train]
ids_test = [[vocab.get(t, 0) for t in d] for d in tok_test]

for dim in (2, 16, 64):
    set_seed()
    clf = ...
    hist = ...
    print(dim, "весов:", ..., "доля верных на тесте:", ...)

**Вывод:**

_напишите одну-две фразы_

## Задача 7. Память рекуррентных сетей

Задача «запомнить первый символ» из второго блокнота. Функция `train_memory(cell, length, steps=1000, lr=3e-3)` обучает модель и возвращает долю верных ответов. Параметр `cell` принимает значения `"rnn"`, `"gru"`, `"lstm"` (инициализация по умолчанию) и `"lstm_open"` (вентиль забывания открыт).

1. Измерьте долю верных ответов четырёх моделей при длине последовательности 15.
2. Для простой RNN найдите перебором наибольшую длину из набора 5, 6, 7, 8, 10, при которой доля верных ответов больше 0.9.

**Самопроверка.** При длине 15 доля верных ответов у `"rnn"` и `"lstm"` меньше 0.5, у `"lstm_open"` и `"gru"` больше 0.95. Наибольшая длина для простой RNN лежит между 5 и 8.

In [ ]:
for cell in ("rnn", "lstm", "lstm_open", "gru"):
    print(cell, ...)

for length in (5, 6, 7, 8, 10):
    print("rnn, длина", length, ...)

**Вывод:**

_напишите одну-две фразы_

## Задача 8. Температура

Языковая модель выдала для трёх возможных следующих символов оценки $z=(2.0,\,1.0,\,0.0)$. Перед выборкой оценки делят на температуру $T$: $p=\mathrm{softmax}(z/T)$.

1. Вычислите распределение при $T=0.5$, $T=1$ и $T=2$.
2. Для каждого распределения вычислите энтропию $H=-\sum_i p_i\ln p_i$. Она измеряет разнообразие выборки.
3. К какому распределению стремится $p$ при $T\to0$ и при $T\to\infty$?

**Самопроверка.** Вероятность самого вероятного символа равна 0.867 при $T=0.5$, 0.665 при $T=1$ и 0.506 при $T=2$. Энтропия растёт с температурой: 0.44, 0.83, 1.02.

In [ ]:
z = np.array([2.0, 1.0, 0.0])
for T in (0.5, 1.0, 2.0):
    p = ...
    H = ...
    print(T, p, H)

**Вывод:**

_напишите одну-две фразы_

## Задача 9. Внимание вручную

У кодировщика три состояния: $h^e_1=(1,0)$, $h^e_2=(0,1)$, $h^e_3=(1,1)$. Состояние декодировщика $h^d=(2,1)$.

1. Вычислите оценки сходства скалярным произведением.
2. Вычислите веса внимания функцией softmax.
3. Вычислите контекстный вектор.
4. Как изменятся веса, если состояние декодировщика умножить на 3?

**Самопроверка.** Оценки $(2,\,1,\,3)$. Веса $(0.245,\,0.090,\,0.665)$. Контекстный вектор $(0.910,\,0.755)$. При умножении на 3 наибольший вес вырастает до 0.950.

In [ ]:
enc = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
dec = np.array([2.0, 1.0])

scores = ...
weights = ...
context = ...
print(scores, weights, context)
print("веса при состоянии 3*dec:", ...)

**Вывод:**

_напишите одну-две фразы_

## Задача 10. Другая запись даты

В третьем блокноте модель переводила «12 ноября 2026» в «2026-11-12». Измените вход: пусть дата записана в порядке «год, месяц, день» с названием месяца в именительном падеже, например «2026, ноябрь 12». Выход прежний: «2026-11-12».

1. Напишите функцию `date_pair_ymd(rng)`, которая возвращает пару строк в новой записи. Список названий месяцев `MONTHS_NOM` задан.
2. Создайте 4000 обучающих и 500 тестовых пар, словарь символов и модель с вниманием (`hidden=128`). Обучите её 8 эпох.
3. Постройте карту внимания для входа «2026, ноябрь 12» функцией `plot_attention`.
4. На какие символы входа модель смотрит при порождении года? Чем карта отличается от карты из третьего блокнота?

**Самопроверка.** Доля точных совпадений на тесте после 8 эпох не меньше 0.9. На карте внимания строки, соответствующие цифрам года, имеют наибольший вес в первых шести столбцах (цифры года и стоящие за ними запятая и пробел) или в последнем столбце (первый шаг).

In [ ]:
MONTHS_NOM = ["январь", "февраль", "март", "апрель", "май", "июнь", "июль", "август", "сентябрь", "октябрь", "ноябрь", "декабрь"]


def date_pair_ymd(rng):
    day, month, year = rng.randint(1, 28), rng.randint(1, 12), rng.randint(1950, 2049)
    return ..., ...


train_pairs = make_pairs(date_pair_ymd, 4000, seed=1)
test_pairs = make_pairs(date_pair_ymd, 500, seed=2)
print(train_pairs[:3])
vocab_d = ...
set_seed()
model = ...
history = ...

text = "2026, ноябрь 12"
out, attn = ...
fig, ax = plt.subplots(figsize=(7, 4.4))
plot_attention(ax, attn, text, out, f"{text} → {out}")
plt.show()

**Вывод:**

_напишите одну-две фразы_

## Перед сдачей

- [ ] Блокнот выполняется целиком без ошибок после перезапуска ядра.
- [ ] Результат каждой задачи сопоставлен с самопроверкой.
- [ ] По каждой задаче записан вывод.